### Import Libraries

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Load data
df = pd.read_csv(r"C:\Users\ragha\OneDrive\Desktop\Projects\Loan Default Risk\accepted_2007_to_2018Q4.csv", low_memory=False)

In [3]:
df.shape

(2260701, 151)

In [4]:
df.columns.tolist()

['id',
 'member_id',
 'loan_amnt',
 'funded_amnt',
 'funded_amnt_inv',
 'term',
 'int_rate',
 'installment',
 'grade',
 'sub_grade',
 'emp_title',
 'emp_length',
 'home_ownership',
 'annual_inc',
 'verification_status',
 'issue_d',
 'loan_status',
 'pymnt_plan',
 'url',
 'desc',
 'purpose',
 'title',
 'zip_code',
 'addr_state',
 'dti',
 'delinq_2yrs',
 'earliest_cr_line',
 'fico_range_low',
 'fico_range_high',
 'inq_last_6mths',
 'mths_since_last_delinq',
 'mths_since_last_record',
 'open_acc',
 'pub_rec',
 'revol_bal',
 'revol_util',
 'total_acc',
 'initial_list_status',
 'out_prncp',
 'out_prncp_inv',
 'total_pymnt',
 'total_pymnt_inv',
 'total_rec_prncp',
 'total_rec_int',
 'total_rec_late_fee',
 'recoveries',
 'collection_recovery_fee',
 'last_pymnt_d',
 'last_pymnt_amnt',
 'next_pymnt_d',
 'last_credit_pull_d',
 'last_fico_range_high',
 'last_fico_range_low',
 'collections_12_mths_ex_med',
 'mths_since_last_major_derog',
 'policy_code',
 'application_type',
 'annual_inc_joint',
 '

In [5]:
df.head()

,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,68407277,NaN,3600.0,3600.0,3600.0,36 months,13.99,123.03,C,C4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
1,68355089,NaN,24700.0,24700.0,24700.0,36 months,11.99,820.28,C,C1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2,68341763,NaN,20000.0,20000.0,20000.0,60 months,10.78,432.66,B,B4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
3,66310712,NaN,35000.0,35000.0,35000.0,60 months,14.85,829.90,C,C5,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
4,68476807,NaN,10400.0,10400.0,10400.0,60 months,22.45,289.91,F,F1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2260701 entries, 0 to 2260700
Columns: 151 entries, id to settlement_term
dtypes: float64(113), object(38)
memory usage: 2.5+ GB


### Data Cleaning & Feature Engineering

In [7]:
# Inspecting duplicates
df.duplicated().sum()

np.int64(0)

In [8]:
# Inspecting loan_status categories
df['loan_status'].value_counts(dropna=False)

loan_status
Fully Paid                                             1076751
Current                                                 878317
Charged Off                                             268559
Late (31-120 days)                                       21467
In Grace Period                                           8436
Late (16-30 days)                                         4349
Does not meet the credit policy. Status:Fully Paid        1988
Does not meet the credit policy. Status:Charged Off        761
Default                                                     40
NaN                                                         33
Name: count, dtype: int64

In [9]:
# Default Flag creation
df['default_flag'] = df['loan_status'].isin([
    'Charged Off',
    'Default',
    'Does not meet the credit policy. Status:Charged Off']).astype(int)

In [10]:
df['default_flag'].value_counts()

default_flag
0    1991341
1     269360
Name: count, dtype: int64

In [11]:
# Inspecting default flag
df.loc[df['default_flag'] == 1, 'loan_status'].value_counts()

loan_status
Charged Off                                            268559
Does not meet the credit policy. Status:Charged Off       761
Default                                                    40
Name: count, dtype: int64

In [12]:
# Select relevant columns for analysis
selected_cols = [
    'id',
    'loan_amnt',
    'term',
    'int_rate',
    'installment',
    'grade',
    'sub_grade',
    'emp_length',
    'home_ownership',
    'annual_inc',
    'verification_status',
    'purpose',
    'dti',
    'fico_range_low',
    'fico_range_high',
    'loan_status',
    'default_flag'
]

loan_df = df[selected_cols].copy()

In [13]:
loan_df.columns.tolist()

['id',
 'loan_amnt',
 'term',
 'int_rate',
 'installment',
 'grade',
 'sub_grade',
 'emp_length',
 'home_ownership',
 'annual_inc',
 'verification_status',
 'purpose',
 'dti',
 'fico_range_low',
 'fico_range_high',
 'loan_status',
 'default_flag']

In [14]:
loan_df.head()

,id,loan_amnt,term,int_rate,installment,grade,sub_grade,emp_length,home_ownership,annual_inc,verification_status,purpose,dti,fico_range_low,fico_range_high,loan_status,default_flag
0,68407277,3600.0,36 months,13.99,123.03,C,C4,10+ years,MORTGAGE,55000.0,Not Verified,debt_consolidation,5.91,675.0,679.0,Fully Paid,0
1,68355089,24700.0,36 months,11.99,820.28,C,C1,10+ years,MORTGAGE,65000.0,Not Verified,small_business,16.06,715.0,719.0,Fully Paid,0
2,68341763,20000.0,60 months,10.78,432.66,B,B4,10+ years,MORTGAGE,63000.0,Not Verified,home_improvement,10.78,695.0,699.0,Fully Paid,0
3,66310712,35000.0,60 months,14.85,829.90,C,C5,10+ years,MORTGAGE,110000.0,Source Verified,debt_consolidation,17.06,785.0,789.0,Current,0
4,68476807,10400.0,60 months,22.45,289.91,F,F1,3 years,MORTGAGE,104433.0,Source Verified,major_purchase,25.37,695.0,699.0,Fully Paid,0


In [15]:
loan_df.shape

(2260701, 17)

In [16]:
# Inspecting missing values
loan_df.isnull().sum().sort_values(ascending=False)

emp_length             146940
dti                      1744
annual_inc                 37
home_ownership             33
loan_status                33
term                       33
int_rate                   33
installment                33
grade                      33
sub_grade                  33
loan_amnt                  33
verification_status        33
purpose                    33
fico_range_low             33
fico_range_high            33
id                          0
default_flag                0
dtype: int64

In [17]:
# Inspect emp length
loan_df['emp_length'].value_counts(dropna=False)

emp_length
10+ years    748005
2 years      203677
< 1 year     189988
3 years      180753
1 year       148403
NaN          146940
5 years      139698
4 years      136605
6 years      102628
7 years       92695
8 years       91914
9 years       79395
Name: count, dtype: int64

In [18]:
# Fill missing values of emp length
loan_df['emp_length'] = loan_df['emp_length'].fillna('Unknown')

In [19]:
loan_df['emp_length'].value_counts(dropna=False)

emp_length
10+ years    748005
2 years      203677
< 1 year     189988
3 years      180753
1 year       148403
Unknown      146940
5 years      139698
4 years      136605
6 years      102628
7 years       92695
8 years       91914
9 years       79395
Name: count, dtype: int64

In [20]:
# Inspect dti
loan_df['dti'].describe()

count    2.258957e+06
mean     1.882420e+01
std      1.418333e+01
min     -1.000000e+00
25%      1.189000e+01
50%      1.784000e+01
75%      2.449000e+01
max      9.990000e+02
Name: dti, dtype: float64

In [21]:
loan_df['dti'].quantile([0, 0.01, 0.05, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 0.995, 0.999, 1])

0.000     -1.00000
0.010      1.72000
0.050      4.94000
0.250     11.89000
0.500     17.84000
0.750     24.49000
0.900     30.50000
0.950     33.88000
0.990     42.71000
0.995     53.68000
0.999    106.63132
1.000    999.00000
Name: dti, dtype: float64

In [22]:
# Fill missing values of dti with median
loan_df['dti'] = loan_df['dti'].fillna(loan_df['dti'].median())

In [23]:
# Remove unsual values of dti
loan_df = loan_df[(loan_df['dti'] > 1) & (loan_df['dti'] <=100)]

In [24]:
loan_df['dti'].isnull().sum()

np.int64(0)

In [25]:
# Drop missing values of loan status
loan_df = loan_df.dropna(subset = ['loan_status'])

In [26]:
loan_df.isna().sum()

id                     0
loan_amnt              0
term                   0
int_rate               0
installment            0
grade                  0
sub_grade              0
emp_length             0
home_ownership         0
annual_inc             1
verification_status    0
purpose                0
dti                    0
fico_range_low         0
fico_range_high        0
loan_status            0
default_flag           0
dtype: int64

In [27]:
# Drop missing values of annual inc
loan_df = loan_df.dropna(subset = ['annual_inc'])

In [28]:
# Income distribution
loan_df['annual_inc'].quantile([0, 0.01, 0.05, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 0.995, 0.999, 1])

0.000          0.0
0.010      17250.0
0.050      28000.0
0.250      46000.0
0.500      65000.0
0.750      93000.0
0.900     130000.0
0.950     160000.0
0.990     269000.0
0.995     340000.0
0.999     590000.0
1.000    5000000.0
Name: annual_inc, dtype: float64

In [29]:
# Keeping values of more than zero income
loan_df = loan_df[loan_df['annual_inc'] > 0]

In [30]:
# Make categories for income

conditions = [
    loan_df['annual_inc'] < 40001,
    loan_df['annual_inc'] < 65001,
    loan_df['annual_inc'] < 100001,
    loan_df['annual_inc'] < 150001,
    loan_df['annual_inc'] < 250001,
    loan_df['annual_inc'] >= 250001,
]

categories = [
    'Low',
    'Lower-Middle',
    'Middle',
    'Upper-Middle',
    'High',
    'Very-High'
]

loan_df['income_category'] = np.select(conditions, categories, default = 'Unknown')

In [31]:
loan_df['income_category'].value_counts()

income_category
Lower-Middle    736518
Middle          659864
Low             404912
Upper-Middle    311306
High            107047
Very-High        25320
Name: count, dtype: int64

In [32]:
loan_df.isna().sum()

id                     0
loan_amnt              0
term                   0
int_rate               0
installment            0
grade                  0
sub_grade              0
emp_length             0
home_ownership         0
annual_inc             0
verification_status    0
purpose                0
dti                    0
fico_range_low         0
fico_range_high        0
loan_status            0
default_flag           0
income_category        0
dtype: int64

In [33]:
loan_df.shape

(2244967, 18)

In [34]:
loan_df.dtypes

id                      object
loan_amnt              float64
term                    object
int_rate               float64
installment            float64
grade                   object
sub_grade               object
emp_length              object
home_ownership          object
annual_inc             float64
verification_status     object
purpose                 object
dti                    float64
fico_range_low         float64
fico_range_high        float64
loan_status             object
default_flag             int64
income_category         object
dtype: object

In [35]:
loan_df.describe()

,loan_amnt,int_rate,installment,annual_inc,dti,fico_range_low,fico_range_high,default_flag
count,2.244967e+06,2.244967e+06,2.244967e+06,2.244967e+06,2.244967e+06,2.244967e+06,2.244967e+06,2.244967e+06
mean,1.503815e+04,1.309470e+01,4.455757e+02,7.750472e+04,1.866230e+01,6.983898e+02,7.023899e+02,1.194276e-01
std,9.174523e+03,4.828342e+00,2.667164e+02,5.502534e+04,9.230522e+00,3.279327e+01,3.279407e+01,3.242911e-01
min,5.000000e+02,5.310000e+00,4.930000e+00,3.600000e-01,1.010000e+00,6.100000e+02,6.140000e+02,0.000000e+00
25%,8.000000e+03,9.490000e+00,2.517500e+02,4.600000e+04,1.198000e+01,6.750000e+02,6.790000e+02,0.000000e+00
50%,1.290000e+04,1.262000e+01,3.779500e+02,6.500000e+04,1.788000e+01,6.900000e+02,6.940000e+02,0.000000e+00
75%,2.000000e+04,1.599000e+01,5.926300e+02,9.300000e+04,2.450000e+01,7.150000e+02,7.190000e+02,0.000000e+00
max,4.000000e+04,3.099000e+01,1.719830e+03,5.000000e+06,1.000000e+02,8.450000e+02,8.500000e+02,1.000000e+00


In [36]:
loan_df['annual_inc'].quantile([0.95, 0.99, 0.995, 0.999, 1])

0.950     160000.00
0.990     269571.62
0.995     340000.00
0.999     590000.00
1.000    5000000.00
Name: annual_inc, dtype: float64

In [37]:
# Create Fico score column
loan_df['fico_score'] = (loan_df['fico_range_low'] + loan_df['fico_range_high']) / 2

In [38]:
loan_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2244967 entries, 0 to 2260698
Data columns (total 19 columns):
 #   Column               Dtype  
---  ------               -----  
 0   id                   object 
 1   loan_amnt            float64
 2   term                 object 
 3   int_rate             float64
 4   installment          float64
 5   grade                object 
 6   sub_grade            object 
 7   emp_length           object 
 8   home_ownership       object 
 9   annual_inc           float64
 10  verification_status  object 
 11  purpose              object 
 12  dti                  float64
 13  fico_range_low       float64
 14  fico_range_high      float64
 15  loan_status          object 
 16  default_flag         int64  
 17  income_category      object 
 18  fico_score           float64
dtypes: float64(8), int64(1), object(10)
memory usage: 342.6+ MB


In [39]:
loan_df.isna().sum()

id                     0
loan_amnt              0
term                   0
int_rate               0
installment            0
grade                  0
sub_grade              0
emp_length             0
home_ownership         0
annual_inc             0
verification_status    0
purpose                0
dti                    0
fico_range_low         0
fico_range_high        0
loan_status            0
default_flag           0
income_category        0
fico_score             0
dtype: int64

In [40]:
# Inspect Categorical columns
categorical_cols = [
    'term',
    'grade',
    'sub_grade',
    'emp_length',
    'home_ownership',
    'verification_status',
    'purpose',
    'loan_status'
]

for col in categorical_cols:
    print(f'\n---{col}---')
    print(loan_df[col].value_counts(dropna=False))


---term---
term
36 months    1598537
60 months     646430
Name: count, dtype: int64

---grade---
grade
B    659215
C    646098
A    428979
D    322323
E    134751
F     41553
G     12048
Name: count, dtype: int64

---sub_grade---
sub_grade
C1    145044
B5    139404
B4    138952
B3    130726
C2    130318
C3    128372
C4    126391
B2    125716
B1    124417
C5    115973
A5    106826
A4     94936
A1     85950
D1     81294
D2     72447
A3     72434
A2     68833
D3     64346
D4     56524
D5     47712
E1     33395
E2     29751
E3     26547
E4     22586
E5     22472
F1     13331
F2      9257
F3      7749
F4      6084
F5      5132
G1      4056
G2      2668
G3      2072
G4      1703
G5      1549
Name: count, dtype: int64

---emp_length---
emp_length
10+ years    744653
2 years      202357
< 1 year     188124
3 years      179639
1 year       147478
Unknown      143223
5 years      138841
4 years      135780
6 years      102109
7 years       92224
8 years       91462
9 years       79077
Name: cou

### Export cleaned dataset

In [41]:
loan_df.to_csv(r"C:\Users\ragha\OneDrive\Desktop\Projects\Loan Default Risk\loan_data_cleaned.csv", index = False)